# Solution 03 — the recipe extractor

A `Recipe` model, an agent, and one `output_validator` enforcing two business rules with
`ModelRetry`. Paste any recipe text to test it.

In [ ]:
# ── SETUP — run this first in every notebook ─────────────────────────────────
import os
import sys

import logfire
from dotenv import find_dotenv, load_dotenv

# 1) Environment: walk up from notebooks/ to the repo root and load .env
load_dotenv(find_dotenv())

# 2) Observability: no Logfire account yet? This still works — data only ships
#    when a token exists (`uv run logfire auth`, or a LOGFIRE_TOKEN in .env).
logfire.configure(send_to_logfire="if-token-present")
logfire.instrument_pydantic_ai()  # v2: instrumentation is explicit — Agent(instrument=True) is gone

MODEL = os.getenv("MODEL", "openai:gpt-6.1-sol")

print("kernel :", sys.executable)  # should end in learningdantic/.venv/bin/python
print("MODEL  :", MODEL)

In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent, ModelRetry


class Recipe(BaseModel):
    title: str
    ingredients: list[str]
    steps: list[str]
    minutes: int


recipe_agent = Agent(
    MODEL,
    output_type=Recipe,
    instructions="Extract the recipe from the user text. Invent nothing.",
)


@recipe_agent.output_validator
def sanity_check(ctx, output: Recipe) -> Recipe:
    """Business rules that the shape alone can't enforce."""
    if not output.ingredients:
        raise ModelRetry("A recipe with zero ingredients is not a recipe. Regenerate.")
    if output.minutes <= 0:
        raise ModelRetry(
            f"{output.minutes} minutes is impossible — regenerate with a positive time."
        )
    if len(output.ingredients) > 10:
        raise ModelRetry("More than 10 ingredients — summarize into core ingredients only.")
    return output


sample = """
French toast: whisk 2 eggs with 100ml milk and a pinch of cinnamon.
Soak 2 slices of brioche, fry in butter about 3 minutes per side. Takes 10 minutes total.
"""

result = await recipe_agent.run(sample)
print(result.output)

**Notes**

- The validator runs *after* Pydantic validates the shape — it's your last word on meaning.
- Each `ModelRetry` costs a round trip. Keep rules that catch likely model mistakes;
  impossible-to-satisfy validators will burn your retry budget and then raise.